In [10]:
# imports
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [11]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Ejecución en local")

# TENER EN CUENTA DESDE DÓNDE ESTAMOS EJECUTANDO PARA ENCONTRAR EL REQUIREMENTS
BASE_PATH = "/content/drive/MyDrive/TFMCode" if IN_COLAB else ".."

if IN_COLAB:
    drive.mount('/content/drive')
    print("Ejecución en colab")

    %pip install -r "{BASE_PATH}/requirements.txt"

Ejecución en local


# Preprocesamiento

En este notebook realizaremos el preprocesamiento necesario a nuestros datasets para que los posteriores modelos puedan procesar los datos adecuadamente. Este preprocesamiento se organiza en tres fases: en primer lugar, la homogeneización y tratamiento de valores nulos; en segundo lugar, la transformación a escala positiva de las señales; y en tercer lugar, el escalado de características.

## Homogeneización y tratamiento de valores nulos

Optamos por homogeneizar nuestras señales de RSS, donde un +100 indica un AP no detectado, sustituyendo por un mínimo técnico constante del valor mínimo presente -1.

In [12]:
def handle_outliers(train_df, test_df):
    """
    Sustituye los valores +100 por el valor mínimo detectado en train -1
    """
    # calculamos el mínimo real ignorando el +100
    min_detected = train_df[train_df != 100].min().min()
    null_constant = min_detected - 1
    
    # sustituimos en ambos conjuntos usando el valor de train
    train_clean = train_df.replace(100, null_constant)
    test_clean = test_df.replace(100, null_constant)
    
    print(f"Valores +100 sustituidos por {null_constant} (min {min_detected} - 1)")
    return train_clean, test_clean, null_constant

## Transformación a escala positiva de las señales

Aplicamos una translación lineal de los valores de RSS para transformar a escala positiva los datos registrados en dBm (escala negativa en el ámbito del Wi-Fi FP)

In [13]:
def to_positive_scale(train_df, test_df, null_constant):
    """
    Transforma las señales a escala positiva restando el valor nulo
    """
    train_pos = train_df - null_constant
    test_pos = test_df - null_constant
    
    print("Transformación a escala positiva aplicada")
    return train_pos, test_pos

## Escalado de características

Vamos a aplicar un escalado min-max al rango [0,1] para escalar nuestros datos

In [14]:
def scale_features(train_df, test_df, method):
    """
    Aplica escalado de características
    """
    if method == 'minmax':
        # calculamos max de train para evitar data leak
        max_val = train_df.max().max()
        # evitamos división por cero si todos los valores fueran iguales
        divisor = max_val if max_val != 0 else 1
        
        train_scaled = train_df / divisor
        test_scaled = test_df / divisor
        print(f"Escalado min-max a [0, 1] realizado")
        
    # posibilidad de añadir otros métodos elif method == 'standard' etc.
    
    return train_scaled, test_scaled

**Función principal de preprocesamiento**

In [15]:
def process_dataset(dataset_name):
    """
    Función principal de preprocesamiento
    """
    # configuramos rutas
    base_path = "../data"
    raw_dir = os.path.join(base_path, "raw", dataset_name)
    clean_dir = os.path.join(base_path, "clean", dataset_name)
    os.makedirs(clean_dir, exist_ok=True)
    
    print(f"\nProcesando: {dataset_name}")
    
    # caragmos archivos RSS
    train_rss = pd.read_csv(os.path.join(raw_dir, f"{dataset_name}_trnrss.csv"), header=None)
    test_rss = pd.read_csv(os.path.join(raw_dir, f"{dataset_name}_tstrss.csv"), header=None)
    
    # llamada a función de tratamiento de +100
    train_clean, test_clean, null_const = handle_outliers(train_rss, test_rss)
    
    # llamada a función de transformación a positivo
    train_pos, test_pos = to_positive_scale(train_clean, test_clean, null_const)
    
    # llamada a función de escalado de características
    train_final, test_final = scale_features(train_pos, test_pos, method='minmax')
    
    # guardamos CSV preprocesados con el sufijo _clean
    train_final.to_csv(os.path.join(clean_dir, f"{dataset_name}_trnrss_clean.csv"), index=False, header=False)
    test_final.to_csv(os.path.join(clean_dir, f"{dataset_name}_tstrss_clean.csv"), index=False, header=False)
    
    # copiamos los archivos de coordenada a clean con el sufijo aunque no se preprocesen
    for suffix in ['trncrd', 'tstcrd']:
        crd = pd.read_csv(os.path.join(raw_dir, f"{dataset_name}_{suffix}.csv"), header=None)
        crd.to_csv(os.path.join(clean_dir, f"{dataset_name}_{suffix}_clean.csv"), index=False, header=False)
    
    print(f"Resultados guardados en: {clean_dir}")
    return train_final, test_final

Llamamos a la función principal de preprocesamiento para realizar las transformaciones a nuestros dos datasets

In [16]:
df_man1_train, df_man1_test = process_dataset("MAN1")
df_uji1_train, df_uji1_test = process_dataset("UJI1")


Procesando: MAN1
Valores +100 sustituidos por -101.0 (min -100.0 - 1)
Transformación a escala positiva aplicada
Escalado min-max a [0, 1] realizado
Resultados guardados en: ../data\clean\MAN1

Procesando: UJI1
Valores +100 sustituidos por -105.0 (min -104.0 - 1)
Transformación a escala positiva aplicada
Escalado min-max a [0, 1] realizado
Resultados guardados en: ../data\clean\UJI1


In [17]:
# mostramos el head de los datasets resultantes
print(f"\n Head MAN1 train: {df_man1_train.head()}")
print(f"\n Head MAN1 test: {df_man1_test.head()}")
print(f"\n Head UJI1 train: {df_uji1_train.head()}")
print(f"\n Head UJI1 test: {df_uji1_test.head()}")


 Head MAN1 train:          0         1         2         3         4         5         6   \
0  0.800000  0.633333  0.450000  0.550000  0.466667  0.366667  0.483333   
1  0.850000  0.766667  0.366667  0.616667  0.533333  0.333333  0.433333   
2  0.816667  0.750000  0.466667  0.000000  0.533333  0.000000  0.500000   
3  0.816667  0.783333  0.433333  0.566667  0.500000  0.000000  0.500000   
4  0.833333  0.750000  0.350000  0.566667  0.483333  0.000000  0.500000   

         7         8         9   ...   18   19   20   21   22   23   24   25  \
0  0.350000  0.266667  0.183333  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
1  0.366667  0.266667  0.200000  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
2  0.366667  0.266667  0.000000  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
3  0.366667  0.250000  0.200000  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
4  0.416667  0.250000  0.216667  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   

    26   27  
0  0.0  0.0  
1  0.0  0.0  
2  0.0 